# BrowserAgent: Applied Tool Use for Computer/GUI Environments

| Property | Value |
|---|---|
| Origin | Anthropic's "Computer Use" agent capability (2024 announcement) — this notebook simplifies the pattern to a deterministic mock browser environment for teaching purposes, rather than real screen/browser control. |

This notebook is an **applied** build on top of the Tool Use pattern from `01_Tool_Use_Agentic_Systems.ipynb` and `03_Tool_Use_Alt.ipynb`. It is not a new agentic architecture -- it is the exact same think -> act -> observe loop, just pointed at a different kind of tool: a **GUI / computer-use environment** instead of a web-search API.

The idea is inspired by the "computer use" style demos in `FareedKhan-dev/all-agentic-architectures` (its `34_computer_use` notebook) and by the "From GUI to real-world environment" appendix in `evoiz/Agentic-Design-Patterns`. Both make the same point: once an agent can call tools, a *browser* (or any GUI) is just another tool surface -- `screenshot()`, `click()`, `type()`, `navigate()` become the agent's hands and eyes, and the ReAct loop stays identical.

**Important -- this is a simulation, not real browser automation.** We do **not** launch a real browser, and this notebook has no Playwright/Selenium/computer-use dependency. Instead we build a small in-process `MockBrowser` class: a plain Python object holding fake "page state" (dicts describing the current page's visible text and clickable element ids). The agent calls the same four tools it would call against a real browser, but every call resolves against deterministic Python logic instead of a live webpage. This keeps the notebook dependency-free, fast, reproducible, and safe to run anywhere (including headless CI) while still teaching the real pattern: **tool use applied to an environment with state, not just a single stateless API call.**

## Phase 0: Setup

### Step 0.1: Imports and LLM

**What we are going to do:**
We only need `langchain_core` (for the `@tool` decorator and message types), `langgraph` (for the state graph), and this repo's `helpers` factory for the LLM. No browser-automation package is installed or required.

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AnyMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from helpers import get_llm

llm = get_llm()

## Phase 1: The Mock Browser -- a deterministic, dependency-free "environment"

**What we are going to do:**
We define `MockBrowser`, a plain Python class that simulates a tiny e-commerce site: a home page with a search box, a search-results page, a product page, and a cart. Its state lives entirely in memory (`current_page`, `pending_query`, `last_results`, `cart`). We give it exactly four methods, mirroring what a real browser-automation tool would expose:

- `screenshot_text()` -- returns a text description of what is "visible" on the current page (standing in for a real screenshot + OCR/accessibility-tree read).
- `click(element_id)` -- clicks a named element (a button, a search result, a link).
- `type_text(element_id, text)` -- types into a named input element.
- `navigate(url)` -- jumps to a top-level page (`home` or `cart`).

Every call is a pure state transition over Python dicts -- no network, no rendering, fully deterministic and inspectable.

In [ ]:
class MockBrowser:
    """A tiny, deterministic stand-in for a real browser.

    It does not render HTML or talk to any network. It only tracks a handful
    of Python dicts describing "pages" and mutates them in response to
    click / type_text / navigate calls -- just enough state to exercise the
    same think -> act -> observe loop a real browser-automation tool needs.
    """

    CATALOG = {
        "wireless mouse": [
            {"id": "prod_101", "name": "Logitech M185 Wireless Mouse", "price": 14.99},
            {"id": "prod_102", "name": "Generic 2.4G Wireless Mouse", "price": 8.49},
        ],
        "usb keyboard": [
            {"id": "prod_201", "name": "Logitech K120 USB Keyboard", "price": 12.99},
        ],
    }

    def __init__(self):
        self.current_page = "home"
        self.pending_query = ""
        self.last_results = []
        self._viewed_product = None
        self.cart = []
        self.log = []  # action history, handy for debugging / eval

    def _record(self, action: str) -> None:
        self.log.append(action)

    # ---- the four tools an agent is allowed to call ------------------------
    def navigate(self, url: str) -> str:
        self._record(f"navigate({url!r})")
        if "cart" in url.lower():
            self.current_page = "cart"
        else:
            self.current_page = "home"
            self.pending_query = ""
            self.last_results = []
        return self.screenshot_text()

    def type_text(self, element_id: str, text: str) -> str:
        self._record(f"type_text({element_id!r}, {text!r})")
        if self.current_page == "home" and element_id == "search_box":
            self.pending_query = text.strip().lower()
        else:
            return (
                f"ERROR: '{element_id}' is not a text input on the current page.\n\n"
                f"{self.screenshot_text()}"
            )
        return self.screenshot_text()

    def click(self, element_id: str) -> str:
        self._record(f"click({element_id!r})")
        if self.current_page == "home" and element_id == "search_button":
            self.last_results = self.CATALOG.get(self.pending_query, [])
            self.current_page = "search_results"
        elif self.current_page == "search_results" and element_id.startswith("result_"):
            idx = int(element_id.split("_")[1])
            if 0 <= idx < len(self.last_results):
                self._viewed_product = self.last_results[idx]
                self.current_page = "product"
            else:
                return f"ERROR: no result at '{element_id}'.\n\n{self.screenshot_text()}"
        elif self.current_page == "product" and element_id == "add_to_cart":
            self.cart.append(self._viewed_product)
        elif self.current_page == "product" and element_id == "back_to_results":
            self.current_page = "search_results"
        elif element_id == "view_cart":
            self.current_page = "cart"
        else:
            return f"ERROR: '{element_id}' is not clickable on the current page.\n\n{self.screenshot_text()}"
        return self.screenshot_text()

    # ---- the agent's "eyes" -------------------------------------------------
    def screenshot_text(self) -> str:
        cart_note = f"[Cart: {len(self.cart)} item(s)]"
        if self.current_page == "home":
            return (
                "PAGE: MockShop Home\n"
                "Visible text: 'Search our catalog.'\n"
                "Clickable/typeable elements: [search_box] (text input), [search_button] (button)\n"
                f"{cart_note}"
            )
        if self.current_page == "search_results":
            if not self.last_results:
                body = f"No results found for '{self.pending_query}'."
                elements = "[search_box], [search_button]"
            else:
                lines = [
                    f"  result_{i}: {p['name']} - ${p['price']}"
                    for i, p in enumerate(self.last_results)
                ]
                body = f"Search results for '{self.pending_query}':\n" + "\n".join(lines)
                elements = ", ".join(f"[result_{i}]" for i in range(len(self.last_results)))
            return f"PAGE: Search Results\n{body}\nClickable elements: {elements}\n{cart_note}"
        if self.current_page == "product":
            p = self._viewed_product
            return (
                f"PAGE: Product - {p['name']}\n"
                f"Visible text: 'Price: ${p['price']}. In stock.'\n"
                "Clickable elements: [add_to_cart], [back_to_results]\n"
                f"{cart_note}"
            )
        if self.current_page == "cart":
            if not self.cart:
                body = "Your cart is empty."
            else:
                body = "Cart contents:\n" + "\n".join(
                    f"  - {p['name']} (${p['price']})" for p in self.cart
                )
            return f"PAGE: Cart\n{body}\nClickable elements: [none]\n{cart_note}"
        return "PAGE: Unknown"


print("MockBrowser defined.")

### Step 1.1: Manually driving the mock browser

**What we are going to do:**
Before wiring any LLM into this, let's confirm the environment itself behaves correctly by manually scripting the exact flow we'll later ask the agent to perform: search for "wireless mouse", open the first result, add it to the cart.

In [ ]:
demo_browser = MockBrowser()

print(demo_browser.screenshot_text())
print("\n---\n")
print(demo_browser.type_text("search_box", "wireless mouse"))
print("\n---\n")
print(demo_browser.click("search_button"))
print("\n---\n")
print(demo_browser.click("result_0"))
print("\n---\n")
print(demo_browser.click("add_to_cart"))
print("\n---\n")
print("Cart contents:", demo_browser.cart)
print("Action log:", demo_browser.log)

**Discussion of the Output:**
Each call returns the next "screenshot" text, exactly like a real browser tool would return a fresh screenshot/DOM read after every action. The `search_box` -> `search_button` -> `result_0` -> `add_to_cart` sequence deterministically lands one `Logitech M185 Wireless Mouse` in the cart, and `demo_browser.log` gives us a clean audit trail of every action taken. This is the environment the agent will operate blind inside of -- it never sees this Python code, only the text `screenshot_text()` returns.

## Phase 2: Exposing the Mock Browser as Tools

**What we are going to do:**
We wrap the four `MockBrowser` methods as LangChain tools with `@tool`. A fresh `browser` instance is created for the run, and each tool function is a thin closure around it. The tool *descriptions* matter a lot here -- they are the only way the agent knows these tools exist and roughly what they do, since (unlike a human) it never sees the actual GUI.

In [ ]:
browser = MockBrowser()


@tool
def screenshot_text() -> str:
    """Take a 'screenshot' of the current page and return its visible text
    plus the ids of every clickable or typeable element on it."""
    return browser.screenshot_text()


@tool
def click(element_id: str) -> str:
    """Click a clickable element by its id (e.g. 'search_button', 'result_0',
    'add_to_cart', 'back_to_results', 'view_cart'). Returns the resulting screenshot text."""
    return browser.click(element_id)


@tool
def type_text(element_id: str, text: str) -> str:
    """Type text into a text-input element by its id (e.g. 'search_box').
    Returns the resulting screenshot text."""
    return browser.type_text(element_id, text)


@tool
def navigate(url: str) -> str:
    """Navigate directly to a top-level page: 'home' or 'cart'.
    Returns the resulting screenshot text."""
    return browser.navigate(url)


browser_tools = [screenshot_text, click, type_text, navigate]
print(f"Defined {len(browser_tools)} browser tools: {[t.name for t in browser_tools]}")

## Phase 3: A ReAct-Style Loop Over the Browser Environment

**What we are going to do:**
This is the same graph shape as `03_Tool_Use_Alt.ipynb` -- `agent -> router -> tool -> agent` -- with two applied-setting additions:

1. The system prompt tells the agent it is operating a GUI blind, purely through `screenshot_text`/`click`/`type_text`/`navigate`, and must call `screenshot_text` whenever it is unsure what is on the page.
2. A `turns` counter in the graph state enforces a hard turn cap, so a confused or looping agent cannot run forever against the mock environment -- mirroring the real safety concern with any computer-use agent that can take unbounded actions.

In [ ]:
MAX_TURNS = 8


class BrowserAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    turns: int


SYSTEM_PROMPT = SystemMessage(content=(
    "You are BrowserAgent, an agent that completes shopping tasks on a website you can "
    "only observe through text. You do not see pixels or HTML -- you only see whatever "
    "the `screenshot_text` tool returns. That text lists the page's visible content and "
    "the element ids you are allowed to `click` or `type_text` into. `navigate` jumps to "
    "a top-level page ('home' or 'cart'). "
    "Always call `screenshot_text` first if you are unsure what is currently on screen. "
    "Work step by step: one tool call at a time, then read the returned screenshot text "
    "before deciding the next action. Once the goal is verifiably achieved (for example, "
    "you can see the item listed in the cart), stop calling tools and reply with a short "
    "final confirmation message instead."
))

llm_with_browser_tools = llm.bind_tools(browser_tools)


def agent_node(state: BrowserAgentState):
    """The 'brain': calls the LLM with the running conversation to decide the next action."""
    messages = state["messages"]
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SYSTEM_PROMPT] + messages
    response = llm_with_browser_tools.invoke(messages)
    return {"messages": [response], "turns": state.get("turns", 0) + 1}


tool_node = ToolNode(browser_tools)


def router_function(state: BrowserAgentState) -> str:
    """Route to the tool node, or end -- whichever comes first: a final answer or the turn cap."""
    if state.get("turns", 0) >= MAX_TURNS:
        return "__end__"
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "call_tool"
    return "__end__"


graph_builder = StateGraph(BrowserAgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("call_tool", tool_node)
graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges("agent", router_function)
graph_builder.add_edge("call_tool", "agent")

browser_agent_app = graph_builder.compile()
print("BrowserAgent graph compiled successfully!")

**Discussion of the Output:**
This is structurally identical to a plain Tool Use agent: one LLM node, one tool-execution node, and a router that loops between them. The only "computer-use" specific pieces are (a) the system prompt framing the tools as a blind GUI interface rather than a search API, and (b) the `turns` counter, which caps how many actions the agent may take against the mock environment before we force a stop -- a cheap but important safety valve for any agent that can take real actions in an environment.

## Phase 4: Running the BrowserAgent

**What we are going to do:**
We reset the shared `browser` instance and give the agent a single goal: *"Search for 'wireless mouse' and add the first result to the cart."* We stream the graph so we can watch each `screenshot_text` observation and each `click`/`type_text` action the agent chooses, then inspect the final `browser.cart` to confirm the goal was actually reached in the environment (not just claimed in text).

In [ ]:
browser.__init__()  # reset the shared mock environment for a clean run

goal = "Search for 'wireless mouse' and add the first result to the cart."
initial_state = {
    "messages": [SYSTEM_PROMPT, ("user", goal)],
    "turns": 0,
}

print(f"Goal: {goal}\n")

for chunk in browser_agent_app.stream(initial_state, stream_mode="values"):
    chunk["messages"][-1].pretty_print()
    print("\n---\n")

print("Final browser.cart:", browser.cart)
print("Full action log:", browser.log)
goal_reached = any("mouse" in p["name"].lower() for p in browser.cart)
print(f"\nGoal reached: {goal_reached}")

**Discussion of the Output:**
A successful trace looks like: the agent calls `screenshot_text` (or goes straight to `type_text`) to see the home page, `type_text("search_box", "wireless mouse")`, `click("search_button")`, reads the results, `click("result_0")`, and finally `click("add_to_cart")`, after which it replies with a plain-text confirmation instead of another tool call. Because `MockBrowser` is fully deterministic, we can grade success objectively by inspecting `browser.cart` directly -- we are not trusting the agent's own claim that it succeeded, the same way a real computer-use evaluation would check the actual application/OS state rather than the agent's narration. If the agent instead wandered (e.g., clicked an id that doesn't exist), `MockBrowser` returns an explicit `ERROR: ...` string plus the current screenshot, giving the agent a chance to self-correct on the next turn -- exactly the kind of recoverable, observable failure a real GUI tool should also surface.

## Conclusion

This notebook did not introduce a new agentic architecture -- it applied the same **Tool Use** pattern from earlier in this folder to a **stateful GUI/computer-use environment** instead of a single stateless API call. Key takeaways:

- **Tool use generalizes to environments, not just APIs.** `screenshot_text`, `click`, `type_text`, and `navigate` are still just tools with names, descriptions, and arguments -- the LLM reasons about them exactly like it reasoned about `web_search` earlier, via the identical `agent -> router -> tool -> agent` loop.
- **A mock environment is enough to teach and test the pattern.** `MockBrowser` needed no browser, no display, and no extra dependency, yet it forces the agent to observe state after every action and adapt -- the core skill real computer-use agents need.
- **Ground truth should live in the environment, not the agent's words.** We verified success by reading `browser.cart` directly rather than trusting the agent's final message, and errors from invalid actions are returned as observations the agent can react to, not silent failures.
- **Turn caps matter once an agent can take actions.** Unlike a read-only search tool, `click`/`type_text` mutate state, so bounding the loop (`MAX_TURNS`) is a cheap but essential safety mechanism.

This notebook is a sibling to `05_SWE_Agent_Applied.ipynb` in this same folder -- both are *applied* Tool Use builds (GUI/computer-use here, software-engineering tasks there) rather than new primitives, consistent with this repo's grouping of `01_Tool_Use/` as "environment-applied Tool Use."